# NSGD + RL Autoscaling — Experiment Analysis

This notebook now reads both middleware outputs from CSV, using the same schema for NSGD and RL runs.

**Expected files:**
- `logs/nsgd_constant_poss_5/request_log.csv` — NSGD per-request data
- `logs/nsgd_constant_poss_5/samples.csv` — NSGD periodic aggregate data
- `logs/rl/request_log.csv` — RL per-request data collected by `rl_app.py`
- `logs/rl/samples.csv` — RL periodic aggregate data collected by `rl_app.py`

The old standalone PPO reward-history JSON is no longer required. The RL comparison cells below use the CSVs generated by the RL middleware.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from pathlib import Path
import os
import glob

plt.rcParams.update({
    'figure.figsize': (10, 5),
    'font.size': 12,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'lines.linewidth': 1.5,
})

# ---------------------------------------------------------------------------
# Log locations
# ---------------------------------------------------------------------------
# Override these from your shell/Jupyter environment if needed:
#   export NSGD_LOG_DIR=logs/my_nsgd_run
#   export RL_LOG_DIR=logs/my_rl_run
NSGD_LOG_DIR = Path(os.getenv('NSGD_LOG_DIR', 'logs/nsgd_constant_poss_5'))
RL_LOG_DIR = Path(os.getenv('RL_LOG_DIR', 'logs/rl'))

# Backward compatibility: the existing NSGD cells save figures using LOG_DIR.
LOG_DIR = str(NSGD_LOG_DIR)
RL_OUTPUT_DIR = str(RL_LOG_DIR)

SAMPLING_WINDOW_SECONDS = int(os.getenv('SAMPLING_WINDOW', '30'))
MAX_REPLICAS = int(os.getenv('MAX_REPLICAS', '24'))

NSGD_LOG_DIR.mkdir(parents=True, exist_ok=True)
RL_LOG_DIR.mkdir(parents=True, exist_ok=True)


def load_experiment_csv(log_dir: Path, filename: str, label: str) -> pd.DataFrame:
    """Load one experiment CSV and attach a datetime column when possible."""
    path = log_dir / filename
    if not path.exists():
        print(f'⚠ {label}: {path} not found; returning an empty DataFrame')
        return pd.DataFrame()

    df = pd.read_csv(path)
    if 'timestamp' in df.columns:
        df['datetime'] = pd.to_datetime(df['timestamp'], unit='s', errors='coerce')
    print(f'✓ {label}: loaded {len(df):,} rows from {path}')
    return df


def split_nsgd_modes(request_df: pd.DataFrame):
    """Return training/evaluation request subsets while tolerating missing data."""
    if request_df.empty or 'mode' not in request_df.columns:
        return pd.DataFrame(), pd.DataFrame()
    return (
        request_df[request_df['mode'] == 'training'].copy(),
        request_df[request_df['mode'] == 'evaluation'].copy(),
    )


def duration_minutes(df: pd.DataFrame) -> float:
    if df.empty or 'timestamp' not in df.columns:
        return 0.0
    return (df['timestamp'].max() - df['timestamp'].min()) / 60


def available(df: pd.DataFrame, cols: list[str]) -> list[str]:
    return [c for c in cols if c in df.columns]


def print_dataset_summary(name: str, request_df: pd.DataFrame, sample_df: pd.DataFrame):
    print(f'\n{name}')
    print('-' * len(name))
    print(f'Request log: {len(request_df):,} rows')
    if not request_df.empty:
        print(f'  Duration: {duration_minutes(request_df):.1f} minutes')
        if 'mode' in request_df.columns:
            print('  Modes:', request_df['mode'].value_counts(dropna=False).to_dict())
    print(f'Samples:     {len(sample_df):,} rows')
    if not sample_df.empty:
        cols = available(sample_df, ['nsgd_cost', 'lstm_ppo_reward', 'throughput', 'replicas', 'avg_cpu', 'avg_mem'])
        print('  Main metric columns:', cols)


# ---------------------------------------------------------------------------
# Load NSGD CSVs
# ---------------------------------------------------------------------------
requests = load_experiment_csv(NSGD_LOG_DIR, 'request_log.csv', 'NSGD request_log.csv')
samples = load_experiment_csv(NSGD_LOG_DIR, 'samples.csv', 'NSGD samples.csv')
training, evaluation = split_nsgd_modes(requests)

# ---------------------------------------------------------------------------
# Load RL CSVs collected by rl_app.py
# ---------------------------------------------------------------------------
rl_requests = load_experiment_csv(RL_LOG_DIR, 'request_log.csv', 'RL request_log.csv')
rl_samples = load_experiment_csv(RL_LOG_DIR, 'samples.csv', 'RL samples.csv')
rl = rl_requests[rl_requests.get('mode', pd.Series(dtype=str)) == 'rl'].copy() if not rl_requests.empty and 'mode' in rl_requests.columns else rl_requests.copy()

print_dataset_summary('NSGD dataset', requests, samples)
print(f'  Training requests:   {len(training):,}')
print(f'  Evaluation requests: {len(evaluation):,}')
print_dataset_summary('RL dataset', rl_requests, rl_samples)

print('\nColumns containing nsgd/reward/lstm fields:')
print('  NSGD samples:', [c for c in samples.columns if any(k in c for k in ['nsgd', 'reward', 'lstm'])])
print('  RL samples:  ', [c for c in rl_samples.columns if any(k in c for k in ['nsgd', 'reward', 'lstm'])])


---
## CSV Schema Check

This verifies that the notebook is reading the new RL CSV files with the same request/sample layout as the NSGD middleware. Missing columns do not always mean the run is invalid, but they tell you which plots will be skipped.

In [ ]:
request_columns_expected = [
    'timestamp', 'latency_seconds', 'rejected', 'status_code', 'in_flight',
    'ready_pods', 'not_ready_pods', 'cold', 'idle_on', 'busy', 'init_free',
    'init_reserved', 'saturated_at_arrival', 'is_cold_start', 'nsgd_cost', 'mode'
]

sample_columns_expected = [
    'timestamp', 'nsgd_cost', 'lstm_ppo_reward', 'throughput', 'replicas',
    'avg_cpu', 'avg_mem', 'requests', 'avg_execution_time',
    'reward_throughput', 'reward_cpu', 'reward_mem', 'reward_replicas'
]


def schema_status(name, df, expected):
    existing = set(df.columns)
    return {
        'dataset': name,
        'rows': len(df),
        'available_expected_cols': sum(c in existing for c in expected),
        'missing_expected_cols': ', '.join([c for c in expected if c not in existing]) or 'none',
    }

schema_report = pd.DataFrame([
    schema_status('NSGD request_log', requests, request_columns_expected),
    schema_status('RL request_log', rl_requests, request_columns_expected),
    schema_status('NSGD samples', samples, sample_columns_expected),
    schema_status('RL samples', rl_samples, sample_columns_expected),
])
display(schema_report)


---
## Figure 2 — θ Convergence Over Iterations

In [ ]:
convergence = (
    training
    .dropna(subset=['iteration', 'theta_base_stock'])
    .groupby('iteration')
    .first()
    [['theta_base_stock', 'theta_base_idle', 'theta_base_exp']]
    .reset_index()
)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col, label, color in zip(
    axes,
    ['theta_base_stock', 'theta_base_idle', 'theta_base_exp'],
    [r'$\theta_{stock}$', r'$\theta_{idle}$', r'$\theta_{exp}$'],
    ['#2196F3', '#4CAF50', '#FF9800'],
):
    ax.plot(convergence['iteration'], convergence[col], color=color, marker='o', markersize=3)
    ax.set_xlabel('Iteration $n$')
    ax.set_ylabel(label)
    ax.set_title(label)

fig.suptitle('Control Parameter Convergence (Figure 2)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(f'{LOG_DIR}/fig2_theta_convergence.png', dpi=150, bbox_inches='tight')
plt.show()

## NSGD Cost Over Time (Training)

In [ ]:
window = min(200, len(training) // 5) if len(training) > 10 else 1

fig, ax = plt.subplots()
ax.plot(
    training['datetime'],
    training['nsgd_cost'].rolling(window=window, min_periods=1).mean(),
    color='#E91E63', alpha=0.8, label=f'Rolling mean (w={window})'
)
ax.scatter(
    training['datetime'], training['nsgd_cost'],
    s=1, alpha=0.1, color='gray', label='Per-request'
)
ax.set_xlabel('Time')
ax.set_ylabel('Cost $C(\\theta, x)$')
ax.set_title('NSGD Cost During Training')
ax.legend()
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig(f'{LOG_DIR}/training_cost_over_time.png', dpi=150, bbox_inches='tight')
plt.show()

## Cost Per Iteration (Averaged)

In [ ]:
cost_per_iter = (
    training
    .dropna(subset=['iteration'])
    .groupby('iteration')['nsgd_cost']
    .agg(['mean', 'std', 'count'])
    .reset_index()
)

fig, ax = plt.subplots()
ax.errorbar(
    cost_per_iter['iteration'], cost_per_iter['mean'],
    yerr=cost_per_iter['std'], fmt='-o', markersize=4,
    capsize=3, color='#673AB7', ecolor='#CE93D8',
)
ax.set_xlabel('Iteration $n$')
ax.set_ylabel('Mean Cost')
ax.set_title('Average Cost Per Iteration')
plt.tight_layout()
plt.savefig(f'{LOG_DIR}/cost_per_iteration.png', dpi=150, bbox_inches='tight')
plt.show()

## Evaluation Cost Distribution
Cost under the learned fixed policy — the number reported in Figures 3/4.

In [ ]:
if len(evaluation) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].hist(evaluation['nsgd_cost'], bins=50, color='#009688', edgecolor='white')
    axes[0].axvline(evaluation['nsgd_cost'].mean(), color='red', linestyle='--',
                    label=f'Mean = {evaluation["nsgd_cost"].mean():.2f}')
    axes[0].set_xlabel('Cost $C(\\theta, x)$')
    axes[0].set_ylabel('Count')
    axes[0].set_title('Evaluation Cost Distribution')
    axes[0].legend()

    axes[1].plot(evaluation['datetime'], evaluation['nsgd_cost'].rolling(50, min_periods=1).mean(),
                color='#009688')
    axes[1].set_xlabel('Time')
    axes[1].set_ylabel('Cost (rolling mean)')
    axes[1].set_title('Evaluation Cost Over Time')
    plt.xticks(rotation=30)

    plt.tight_layout()
    plt.savefig(f'{LOG_DIR}/evaluation_cost.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f'Evaluation cost: mean={evaluation["nsgd_cost"].mean():.2f}, '
          f'std={evaluation["nsgd_cost"].std():.2f}, n={len(evaluation)}')
else:
    print('No evaluation data. Run: curl -X POST http://localhost:8000/mode/evaluate')

---
## LSTM-PPO Reward — Computed on NSGD System State
The same reward function from `env.py`, evaluated on what NSGD produced.
Higher is better (opposite of NSGD cost where lower is better).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Reward over time
axes[0].plot(samples['datetime'], samples['lstm_ppo_reward'],
             color='#FF5722', marker='.', markersize=3, alpha=0.7)
axes[0].axhline(samples['lstm_ppo_reward'].mean(), color='red', linestyle='--', alpha=0.6,
                label=f'Mean = {samples["lstm_ppo_reward"].mean():.1f}')
axes[0].set_xlabel('Time')
axes[0].set_ylabel('LSTM-PPO Reward')
axes[0].set_title('LSTM-PPO Reward (computed on NSGD state)')
axes[0].legend()
plt.sca(axes[0])
plt.xticks(rotation=30)

# Reward component breakdown
components = samples[['reward_throughput', 'reward_cpu', 'reward_mem', 'reward_replicas']].mean()
colors = ['#4CAF50', '#2196F3', '#9C27B0', '#F44336']
labels = [f'Throughput\n{components["reward_throughput"]:.1f}',
          f'CPU\n{components["reward_cpu"]:.1f}',
          f'Memory\n{components["reward_mem"]:.1f}',
          f'Replicas\n{components["reward_replicas"]:.1f}']
vals = components.values
bar_colors = [c if v >= 0 else '#F44336' for v, c in zip(vals, colors)]
axes[1].bar(labels, vals, color=bar_colors, edgecolor='white')
axes[1].axhline(0, color='black', linewidth=0.5)
axes[1].set_ylabel('Reward Component')
axes[1].set_title('Avg Reward Breakdown')

plt.tight_layout()
plt.savefig(f'{LOG_DIR}/lstm_ppo_reward.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'LSTM-PPO reward on NSGD system: mean={samples["lstm_ppo_reward"].mean():.2f}, '
      f'std={samples["lstm_ppo_reward"].std():.2f}')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(samples['datetime'], samples['throughput'],
        color='#4CAF50', marker='o', markersize=4)
ax.set_xlabel('Time')
ax.set_ylabel('Throughput (%)')
ax.set_title('Throughput Over Time')
ax.set_ylim(0, 105)
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig(f'{LOG_DIR}/throughput_over_time.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Mean throughput: {samples["throughput"].mean():.2f}%')
print(f'Min:  {samples["throughput"].min():.2f}%')
print(f'Max:  {samples["throughput"].max():.2f}%')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(samples['datetime'], samples['lstm_ppo_reward'],
        color='#FF5722')
ax.set_xlabel('Time')
ax.set_ylabel('LSTM-PPO Reward')
ax.set_title('LSTM-PPO Reward Over Time')
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig(f'{LOG_DIR}/reward_over_time.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Mean reward: {samples["lstm_ppo_reward"].mean():.2f}')
print(f'Min:  {samples["lstm_ppo_reward"].min():.2f}')
print(f'Max:  {samples["lstm_ppo_reward"].max():.2f}')

In [ ]:
if samples.empty:
    print('No NSGD samples available.')
elif {'requests', 'avg_execution_time'}.issubset(samples.columns):
    samples['lambda_over_N_actual'] = (
        (samples['requests'] / SAMPLING_WINDOW_SECONDS) *
        samples['avg_execution_time'] / MAX_REPLICAS
    )
    display(samples[['datetime', 'requests', 'avg_execution_time', 'replicas', 'lambda_over_N_actual']].tail(10))
else:
    print('Missing columns needed for lambda_over_N_actual:', {'requests', 'avg_execution_time'} - set(samples.columns))


In [ ]:
# Look at θ_base over the last 20 iterations
last_iters = (
    training
    .dropna(subset=['iteration', 'theta_base_stock'])
    .groupby('iteration')
    .first()
    .tail(9)
)

print("θ stability over last 20 iterations:")
for col in ['theta_base_stock', 'theta_base_idle', 'theta_base_exp']:
    mean = last_iters[col].mean()
    std = last_iters[col].std()
    cv = std / mean if mean != 0 else float('inf')
    print(f"  {col}: mean={mean:.3f}, std={std:.3f}, CV={cv:.3f}")

## Dual-Axis: NSGD Cost vs LSTM-PPO Reward Over Time
Both metrics on the same timeline. NSGD wants cost DOWN, PPO wants reward UP.

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 5))

ax1.set_xlabel('Time')
ax1.set_ylabel('NSGD Cost (lower = better)', color='#E91E63')
ax1.plot(samples['datetime'], samples['nsgd_cost'], color='#E91E63',
         alpha=0.7, marker='.', markersize=3, label='NSGD Cost')
ax1.tick_params(axis='y', labelcolor='#E91E63')

ax2 = ax1.twinx()
ax2.set_ylabel('LSTM-PPO Reward (higher = better)', color='#2196F3')
ax2.plot(samples['datetime'], samples['lstm_ppo_reward'], color='#2196F3',
         alpha=0.7, marker='.', markersize=3, label='PPO Reward')
ax2.tick_params(axis='y', labelcolor='#2196F3')

fig.suptitle('NSGD Cost vs LSTM-PPO Reward (Same System)', fontsize=14)
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig(f'{LOG_DIR}/nsgd_vs_ppo_dual_axis.png', dpi=150, bbox_inches='tight')
plt.show()

## NSGD vs RL Agent — CSV Comparison

The RL middleware now writes `samples.csv` and `request_log.csv`, so this comparison uses the collected RL CSV data directly instead of the old PPO reward-history JSON.

In [ ]:
if rl_samples.empty:
    print(f'RL samples not found or empty at {RL_LOG_DIR / "samples.csv"}. Run rl_app.py and the RL agent first.')
elif samples.empty:
    print(f'NSGD samples not found or empty at {NSGD_LOG_DIR / "samples.csv"}. Only RL-specific cells can run.')
else:
    metrics_to_compare = available(samples, ['nsgd_cost', 'lstm_ppo_reward', 'throughput', 'replicas', 'avg_cpu', 'avg_mem'])
    metrics_to_compare = [m for m in metrics_to_compare if m in rl_samples.columns]

    if not metrics_to_compare:
        print('No common metric columns found between NSGD and RL samples.')
    else:
        summary_rows = []
        for label, df in [('NSGD', samples), ('RL', rl_samples)]:
            row = {'agent': label, 'samples': len(df)}
            for metric in metrics_to_compare:
                row[f'{metric}_mean'] = df[metric].mean()
                row[f'{metric}_std'] = df[metric].std()
            summary_rows.append(row)

        comparison_summary = pd.DataFrame(summary_rows).set_index('agent')
        display(comparison_summary.round(3))

        # Plot the two most important objectives together when available.
        plot_metrics = [m for m in ['nsgd_cost', 'lstm_ppo_reward', 'throughput', 'replicas'] if m in metrics_to_compare]
        for metric in plot_metrics:
            fig, ax = plt.subplots(figsize=(12, 5))
            ax.plot(range(len(samples)), samples[metric], alpha=0.7, label='NSGD')
            ax.plot(range(len(rl_samples)), rl_samples[metric], alpha=0.7, label='RL')
            ax.set_xlabel('Sample index')
            ax.set_ylabel(metric)
            ax.set_title(f'NSGD vs RL — {metric}')
            ax.legend()
            plt.tight_layout()
            plt.savefig(f'{RL_OUTPUT_DIR}/compare_nsgd_rl_{metric}.png', dpi=150, bbox_inches='tight')
            plt.show()


---
## RL Agent Collected Data — Request Log

These cells use `logs/rl/request_log.csv`. This is the per-request data collected passively by `rl_app.py`: latency, rejection, cold-start flag, pod-state fields, and NSGD equation-7 cost.

In [ ]:
if rl_requests.empty:
    print(f'No RL request log found at {RL_LOG_DIR / "request_log.csv"}.')
else:
    display_cols = available(rl_requests, [
        'datetime', 'mode', 'latency_seconds', 'rejected', 'status_code',
        'in_flight', 'ready_pods', 'not_ready_pods', 'cold', 'idle_on',
        'busy', 'init_free', 'init_reserved', 'is_cold_start', 'nsgd_cost'
    ])
    display(rl_requests[display_cols].head())
    display(rl_requests[display_cols].tail())

    numeric_cols = available(rl_requests, [
        'latency_seconds', 'nsgd_cost', 'in_flight', 'ready_pods', 'not_ready_pods',
        'cold', 'idle_on', 'busy', 'init_free', 'init_reserved'
    ])
    display(rl_requests[numeric_cols].describe().T.round(4))


In [ ]:
if rl_requests.empty:
    print('Skipping RL request-level plots because rl_requests is empty.')
else:
    window = min(200, max(1, len(rl_requests) // 5)) if len(rl_requests) > 10 else 1

    fig, axes = plt.subplots(1, 3, figsize=(18, 4))

    if 'nsgd_cost' in rl_requests.columns:
        axes[0].plot(
            rl_requests['datetime'],
            rl_requests['nsgd_cost'].rolling(window, min_periods=1).mean(),
            label=f'rolling mean w={window}'
        )
        axes[0].set_title('RL Request NSGD Cost')
        axes[0].set_ylabel('Cost')
        axes[0].legend()

    if 'latency_seconds' in rl_requests.columns:
        latency_ms = rl_requests['latency_seconds'] * 1000
        axes[1].plot(rl_requests['datetime'], latency_ms.rolling(window, min_periods=1).mean())
        axes[1].set_title('RL Request Latency')
        axes[1].set_ylabel('Latency (ms)')

    if {'is_cold_start', 'rejected'}.issubset(rl_requests.columns):
        axes[2].plot(
            rl_requests['datetime'],
            rl_requests['is_cold_start'].astype(float).rolling(window, min_periods=1).mean(),
            label='cold-start rate'
        )
        axes[2].plot(
            rl_requests['datetime'],
            rl_requests['rejected'].astype(float).rolling(window, min_periods=1).mean(),
            label='rejection rate'
        )
        axes[2].yaxis.set_major_formatter(ticker.PercentFormatter(1.0))
        axes[2].set_title('RL Cold Starts / Rejections')
        axes[2].set_ylabel('Rate')
        axes[2].legend()

    for ax in axes:
        ax.set_xlabel('Time')
        plt.sca(ax)
        plt.xticks(rotation=30)

    plt.tight_layout()
    plt.savefig(f'{RL_OUTPUT_DIR}/rl_request_log_overview.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f'RL request count: {len(rl_requests):,}')
    if 'latency_seconds' in rl_requests.columns:
        latency_ms = rl_requests['latency_seconds'] * 1000
        print(f'Latency ms: mean={latency_ms.mean():.2f}, p95={latency_ms.quantile(0.95):.2f}, p99={latency_ms.quantile(0.99):.2f}')
    if 'is_cold_start' in rl_requests.columns:
        print(f'Cold-start rate: {rl_requests["is_cold_start"].mean():.4f}')
    if 'rejected' in rl_requests.columns:
        print(f'Rejection rate:  {rl_requests["rejected"].mean():.4f}')


In [ ]:
if rl_requests.empty:
    print('Skipping RL state breakdown because rl_requests is empty.')
else:
    state_cols = available(rl_requests, ['cold', 'idle_on', 'busy', 'init_free', 'init_reserved'])
    if len(state_cols) < 2 or 'datetime' not in rl_requests.columns:
        print('Missing state columns for RL state breakdown:', state_cols)
    else:
        resampled_rl = rl_requests.set_index('datetime')[state_cols].resample('5s').mean().dropna()
        fig, ax = plt.subplots(figsize=(12, 5))
        ax.stackplot(resampled_rl.index, *[resampled_rl[c] for c in state_cols], labels=state_cols, alpha=0.8)
        ax.set_xlabel('Time')
        ax.set_ylabel('Function Instances')
        ax.set_title('RL System State Breakdown')
        ax.legend(loc='upper right')
        plt.xticks(rotation=30)
        plt.tight_layout()
        plt.savefig(f'{RL_OUTPUT_DIR}/rl_state_breakdown.png', dpi=150, bbox_inches='tight')
        plt.show()


---
## RL Agent Collected Data — Periodic Samples

These cells use `logs/rl/samples.csv`. This is the aggregated data collected every sampling window: cost, RL-style reward, throughput, replicas, CPU, memory, and reward components.

In [ ]:
if rl_samples.empty:
    print(f'No RL samples found at {RL_LOG_DIR / "samples.csv"}.')
else:
    display_cols = available(rl_samples, [
        'datetime', 'nsgd_cost', 'lstm_ppo_reward', 'throughput', 'replicas',
        'avg_cpu', 'avg_mem', 'requests', 'avg_execution_time',
        'reward_throughput', 'reward_cpu', 'reward_mem', 'reward_replicas'
    ])
    display(rl_samples[display_cols].head())
    display(rl_samples[display_cols].tail())
    display(rl_samples[display_cols].describe(include='all').T)


In [ ]:
if rl_samples.empty:
    print('Skipping RL sample plots because rl_samples is empty.')
else:
    metrics = available(rl_samples, ['nsgd_cost', 'lstm_ppo_reward', 'throughput', 'replicas', 'avg_cpu', 'avg_mem'])
    for metric in metrics:
        fig, ax = plt.subplots(figsize=(12, 5))
        ax.plot(rl_samples['datetime'], rl_samples[metric], marker='.', alpha=0.8)
        ax.axhline(rl_samples[metric].mean(), linestyle='--', alpha=0.6, label=f'mean={rl_samples[metric].mean():.3f}')
        ax.set_xlabel('Time')
        ax.set_ylabel(metric)
        ax.set_title(f'RL Samples — {metric}')
        ax.legend()
        plt.xticks(rotation=30)
        plt.tight_layout()
        plt.savefig(f'{RL_OUTPUT_DIR}/rl_samples_{metric}.png', dpi=150, bbox_inches='tight')
        plt.show()


In [ ]:
if rl_samples.empty:
    print('Skipping RL reward-component plot because rl_samples is empty.')
else:
    reward_cols = available(rl_samples, ['reward_throughput', 'reward_cpu', 'reward_mem', 'reward_replicas'])
    if not reward_cols:
        print('No reward component columns found in rl_samples.')
    else:
        components = rl_samples[reward_cols].mean().sort_values(ascending=False)
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.bar(components.index, components.values)
        ax.axhline(0, linewidth=0.8)
        ax.set_ylabel('Mean component value')
        ax.set_title('RL Reward Component Breakdown')
        plt.xticks(rotation=20)
        plt.tight_layout()
        plt.savefig(f'{RL_OUTPUT_DIR}/rl_reward_components.png', dpi=150, bbox_inches='tight')
        plt.show()
        display(components.to_frame('mean').round(3))


---
## Final NSGD vs RL Summary Table

This gives one compact table from the collected request logs and one from the collected sample logs.

In [ ]:
def request_summary(df: pd.DataFrame, label: str) -> dict | None:
    if df.empty:
        return None
    row = {'agent': label, 'requests': len(df)}
    if 'nsgd_cost' in df.columns:
        row['mean_request_cost'] = df['nsgd_cost'].mean()
    if 'latency_seconds' in df.columns:
        row['mean_latency_ms'] = df['latency_seconds'].mean() * 1000
        row['p95_latency_ms'] = df['latency_seconds'].quantile(0.95) * 1000
        row['p99_latency_ms'] = df['latency_seconds'].quantile(0.99) * 1000
    if 'is_cold_start' in df.columns:
        row['cold_start_rate'] = df['is_cold_start'].mean()
    if 'rejected' in df.columns:
        row['rejection_rate'] = df['rejected'].mean()
    if 'ready_pods' in df.columns:
        row['mean_ready_pods'] = df['ready_pods'].mean()
    return row

request_rows = [r for r in [request_summary(requests, 'NSGD'), request_summary(rl_requests, 'RL')] if r is not None]
if request_rows:
    display(pd.DataFrame(request_rows).set_index('agent').round(4))
else:
    print('No request logs available.')


def sample_summary(df: pd.DataFrame, label: str) -> dict | None:
    if df.empty:
        return None
    row = {'agent': label, 'samples': len(df)}
    for col in ['nsgd_cost', 'lstm_ppo_reward', 'throughput', 'replicas', 'avg_cpu', 'avg_mem']:
        if col in df.columns:
            row[f'mean_{col}'] = df[col].mean()
    return row

sample_rows = [r for r in [sample_summary(samples, 'NSGD'), sample_summary(rl_samples, 'RL')] if r is not None]
if sample_rows:
    display(pd.DataFrame(sample_rows).set_index('agent').round(4))
else:
    print('No sample logs available.')


---
## Replica Count Over Time

In [ ]:
fig, ax = plt.subplots()

ax.plot(samples['datetime'], samples['replicas'], color='#3F51B5',)
ax.fill_between(samples['datetime'], 0, samples['replicas'], alpha=0.15, color='#3F51B5')
ax.set_xlabel('Time')
ax.set_ylabel('Ready Replicas')
ax.set_title('Replica Count Over Time')
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig(f'{LOG_DIR}/replicas_over_time.png', dpi=150, bbox_inches='tight')
plt.show()

## Cold Start and Rejection Rates

In [ ]:
data = requests.copy()
window = min(200, len(data) // 5) if len(data) > 10 else 1

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

cold_start_roll = data['is_cold_start'].astype(float).rolling(window, min_periods=1).mean()
axes[0].plot(data['datetime'], cold_start_roll, color='#FF5722')
axes[0].set_xlabel('Time')
axes[0].set_ylabel('Cold Start Probability')
axes[0].set_title(f'Cold Start Rate (rolling w={window})')
axes[0].yaxis.set_major_formatter(ticker.PercentFormatter(1.0))

rejection_roll = data['rejected'].astype(float).rolling(window, min_periods=1).mean()
axes[1].plot(data['datetime'], rejection_roll, color='#F44336')
axes[1].set_xlabel('Time')
axes[1].set_ylabel('Rejection Probability')
axes[1].set_title(f'Rejection Rate (rolling w={window})')
axes[1].yaxis.set_major_formatter(ticker.PercentFormatter(1.0))

for ax in axes:
    plt.sca(ax)
    plt.xticks(rotation=30)

plt.tight_layout()
plt.savefig(f'{LOG_DIR}/cold_start_rejection.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Cold start rate: {data["is_cold_start"].mean():.4f} ({data["is_cold_start"].sum()} / {len(data)})')
print(f'Rejection rate:  {data["rejected"].mean():.4f} ({data["rejected"].sum()} / {len(data)})')

## Response Latency Distribution

In [ ]:
latency_ms = requests['latency_seconds'] * 1000

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(latency_ms, bins=80, color='#00BCD4', edgecolor='white', alpha=0.8)
axes[0].axvline(latency_ms.median(), color='red', linestyle='--',
                label=f'Median = {latency_ms.median():.1f}ms')
axes[0].axvline(latency_ms.quantile(0.99), color='orange', linestyle='--',
                label=f'p99 = {latency_ms.quantile(0.99):.1f}ms')
axes[0].set_xlabel('Latency (ms)')
axes[0].set_ylabel('Count')
axes[0].set_title('Latency Distribution')
axes[0].legend()

window = min(100, len(requests) // 5) if len(requests) > 10 else 1
axes[1].plot(requests['datetime'], latency_ms.rolling(window, min_periods=1).mean(),
             color='#00BCD4', alpha=0.8)
axes[1].set_xlabel('Time')
axes[1].set_ylabel('Latency (ms)')
axes[1].set_title(f'Average Latency Over Time (rolling w={window})')
plt.sca(axes[1])
plt.xticks(rotation=30)

plt.tight_layout()
plt.savefig(f'{LOG_DIR}/latency.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Latency: mean={latency_ms.mean():.1f}ms, median={latency_ms.median():.1f}ms, '
      f'p95={latency_ms.quantile(0.95):.1f}ms, p99={latency_ms.quantile(0.99):.1f}ms')

## Throughput and Execution Time (env.py Observables)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(samples['datetime'], samples['avg_execution_time'] * 1000, color='#795548', marker='.')
axes[0].set_ylabel('Avg Execution Time (ms)')
axes[0].set_title('Function Execution Time')

axes[1].plot(samples['datetime'], samples['throughput'], color='#4CAF50', marker='.')
axes[1].set_ylabel('Throughput (%)')
axes[1].set_title('Throughput (% Successful)')
axes[1].set_ylim(0, 105)

axes[2].bar(samples['datetime'], samples['requests'], width=0.001, color='#607D8B')
axes[2].set_ylabel('Requests per Window')
axes[2].set_title('Request Count (per sampling window)')

for ax in axes:
    ax.set_xlabel('Time')
    plt.sca(ax)
    plt.xticks(rotation=30)

plt.tight_layout()
plt.savefig(f'{LOG_DIR}/env_observables.png', dpi=150, bbox_inches='tight')
plt.show()

## CPU and Memory Utilization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(samples['datetime'], samples['avg_cpu'], color='#E91E63', marker='.')
axes[0].set_ylabel('Normalized CPU')
axes[0].set_title('Avg CPU Utilization per Pod')
axes[0].axhline(1.0, color='red', linestyle=':', alpha=0.5, label='100% of request')
axes[0].legend()

axes[1].plot(samples['datetime'], samples['avg_mem'], color='#9C27B0', marker='.')
axes[1].set_ylabel('Normalized Memory')
axes[1].set_title('Avg Memory Utilization per Pod')
axes[1].axhline(1.0, color='red', linestyle=':', alpha=0.5, label='100% of request')
axes[1].legend()

for ax in axes:
    ax.set_xlabel('Time')
    plt.sca(ax)
    plt.xticks(rotation=30)

plt.tight_layout()
plt.savefig(f'{LOG_DIR}/cpu_mem.png', dpi=150, bbox_inches='tight')
plt.show()

## NSGD State Breakdown Over Time

In [ ]:
state_cols = ['cold', 'idle_on', 'busy', 'init_free', 'init_reserved']
resampled = requests.set_index('datetime')[state_cols].resample('5s').mean().dropna()

fig, ax = plt.subplots(figsize=(12, 5))
colors = ['#BDBDBD', '#4CAF50', '#2196F3', '#FFC107', '#F44336']
ax.stackplot(
    resampled.index, *[resampled[c] for c in state_cols],
    labels=['Cold', 'Idle-On', 'Busy', 'Init-Free', 'Init-Reserved'],
    colors=colors, alpha=0.8,
)
ax.set_xlabel('Time')
ax.set_ylabel('Function Instances')
ax.set_title('System State Breakdown')
ax.legend(loc='upper right')
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig(f'{LOG_DIR}/state_breakdown.png', dpi=150, bbox_inches='tight')
plt.show()

## Plus vs Minus Phase Cost

In [ ]:
phase_cost = (
    training
    .dropna(subset=['iteration', 'phase'])
    .groupby(['iteration', 'phase'])['nsgd_cost']
    .mean()
    .unstack(fill_value=np.nan)
    .reset_index()
)

fig, ax = plt.subplots()
if 'plus' in phase_cost.columns:
    ax.plot(phase_cost['iteration'], phase_cost['plus'],
            marker='o', markersize=4, label='Plus phase', color='#4CAF50')
if 'minus' in phase_cost.columns:
    ax.plot(phase_cost['iteration'], phase_cost['minus'],
            marker='s', markersize=4, label='Minus phase', color='#F44336')
ax.set_xlabel('Iteration $n$')
ax.set_ylabel('Mean Cost')
ax.set_title('Cost: Plus vs Minus Perturbation Phases')
ax.legend()
plt.tight_layout()
plt.savefig(f'{LOG_DIR}/phase_cost_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## θ Step Values Applied (Perturbed)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col, label, color in zip(
    axes,
    ['theta_stock', 'theta_idle', 'theta_exp'],
    [r'$\pi_{\theta_{stock}}$', r'$\pi_{\theta_{idle}}$', r'$\theta_{exp}$ (step)'],
    ['#2196F3', '#4CAF50', '#FF9800'],
):
    valid = training.dropna(subset=[col])
    ax.scatter(valid['datetime'], valid[col], s=2, alpha=0.3, color=color)
    ax.set_xlabel('Time')
    ax.set_ylabel(label)
    ax.set_title(label)
    plt.sca(ax)
    plt.xticks(rotation=30)

fig.suptitle('Applied θ (Perturbed) During Training', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(f'{LOG_DIR}/theta_step_applied.png', dpi=150, bbox_inches='tight')
plt.show()

## Periodic Samples — NSGD Cost

In [ ]:
fig, ax = plt.subplots()
ax.plot(samples['datetime'], samples['nsgd_cost'], marker='o', markersize=4, color='#E91E63')
ax.fill_between(samples['datetime'], 0, samples['nsgd_cost'], alpha=0.1, color='#E91E63')
ax.set_xlabel('Time')
ax.set_ylabel('NSGD Cost')
ax.set_title('NSGD Cost (Periodic Samples)')
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig(f'{LOG_DIR}/periodic_cost.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Summary Statistics Table

In [ ]:
def compute_stats(df, label):
    if len(df) == 0:
        return None
    return {
        'Phase': label,
        'Requests': len(df),
        'Mean Cost': round(df['nsgd_cost'].mean(), 2),
        'Std Cost': round(df['nsgd_cost'].std(), 2),
        'Cold Start %': round(df['is_cold_start'].mean() * 100, 2),
        'Rejection %': round(df['rejected'].mean() * 100, 2),
        'Mean Latency (ms)': round(df['latency_seconds'].mean() * 1000, 1),
        'p99 Latency (ms)': round(df['latency_seconds'].quantile(0.99) * 1000, 1),
        'Mean Replicas': round(df['ready_pods'].mean(), 1),
    }

# Per-request stats
rows = [s for s in [compute_stats(training, 'Training'),
                    compute_stats(evaluation, 'Evaluation')] if s]
request_stats = pd.DataFrame(rows).set_index('Phase')
print('=== Per-Request Statistics ===')
display(request_stats)

# Periodic sample stats (includes PPO reward)
print()
print('=== Periodic Sample Statistics ===')
periodic_stats = pd.DataFrame([{
    'Samples': len(samples),
    'Mean NSGD Cost': round(samples['nsgd_cost'].mean(), 2),
    'Std NSGD Cost': round(samples['nsgd_cost'].std(), 2),
    'Mean PPO Reward': round(samples['lstm_ppo_reward'].mean(), 2),
    'Std PPO Reward': round(samples['lstm_ppo_reward'].std(), 2),
    'Mean Throughput %': round(samples['throughput'].mean(), 1),
    'Mean Replicas': round(samples['replicas'].mean(), 1),
    'Mean CPU': round(samples['avg_cpu'].mean(), 3),
    'Mean Mem': round(samples['avg_mem'].mean(), 3),
}])
display(periodic_stats)

## Final θ Values

In [ ]:
last_training = training.dropna(subset=['theta_base_stock']).tail(1)

if len(last_training) > 0:
    row = last_training.iloc[0]
    print('Final learned θ (un-perturbed):')
    print(f'  θ_stock = {row["theta_base_stock"]:.4f}')
    print(f'  θ_idle  = {row["theta_base_idle"]:.4f}')
    print(f'  θ_exp   = {row["theta_base_exp"]:.4f}')
    print(f'  Iteration: {int(row["iteration"])}')
    print()
    if len(evaluation) > 0:
        print(f'Evaluation NSGD cost:  {evaluation["nsgd_cost"].mean():.2f} ± {evaluation["nsgd_cost"].std():.2f}')
        eval_samples = samples[samples['timestamp'] >= evaluation['timestamp'].min()]
        if len(eval_samples) > 0:
            print(f'Evaluation PPO reward: {eval_samples["lstm_ppo_reward"].mean():.2f} ± {eval_samples["lstm_ppo_reward"].std():.2f}')
    else:
        print('No evaluation data. Run: curl -X POST http://localhost:8000/mode/evaluate')
else:
    print('No training data with theta values found.')

## Saved Figures

In [ ]:
saved_nsgd = sorted(glob.glob(f'{LOG_DIR}/*.png'))
saved_rl = sorted(glob.glob(f'{RL_OUTPUT_DIR}/*.png'))
print(f'Saved {len(saved_nsgd)} NSGD figures to {LOG_DIR}/:')
for f in saved_nsgd:
    print(f'  {f}')
print()
print(f'Saved {len(saved_rl)} RL/comparison figures to {RL_OUTPUT_DIR}/:')
for f in saved_rl:
    print(f'  {f}')


In [ ]:
# Find when evaluation mode started (from request_log)
eval_start = requests[requests['mode'] == 'evaluation']['timestamp'].min()

if pd.isna(eval_start):
    print("⚠ No evaluation data found. Run: curl -X POST http://localhost:8000/mode/evaluate")
    eval_samples = pd.DataFrame()
else:
    eval_samples = samples[samples['timestamp'] >= eval_start].copy()
    print(f"Evaluation period starts at: {pd.to_datetime(eval_start, unit='s')}")
    print(f"Evaluation samples: {len(eval_samples)}")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(eval_samples['datetime'], eval_samples['lstm_ppo_reward'],
        color='#FF5722')
ax.set_xlabel('Time')
ax.set_ylabel('LSTM-PPO Reward')
ax.set_title('LSTM-PPO Reward During Evaluation')
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig(f'{LOG_DIR}/eval_lstm_ppo_reward.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Mean: {eval_samples["lstm_ppo_reward"].mean():.2f}')
print(f'Min:  {eval_samples["lstm_ppo_reward"].min():.2f}')
print(f'Max:  {eval_samples["lstm_ppo_reward"].max():.2f}')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(eval_samples['datetime'], eval_samples['throughput'],
        color='#4CAF50')
ax.set_xlabel('Time')
ax.set_ylabel('Throughput (%)')
ax.set_title('Throughput During Evaluation')
ax.set_ylim(0, 105)
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig(f'{LOG_DIR}/eval_throughput.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Mean: {eval_samples["throughput"].mean():.2f}%')
print(f'Min:  {eval_samples["throughput"].min():.2f}%')
print(f'Max:  {eval_samples["throughput"].max():.2f}%')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(eval_samples['datetime'], eval_samples['avg_cpu'],
        color='#E91E63')
ax.set_xlabel('Time')
ax.set_ylabel('Normalized CPU')
ax.set_title('CPU Utilization During Evaluation')
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig(f'{LOG_DIR}/eval_cpu.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Mean: {eval_samples["avg_cpu"].mean():.4f}')
print(f'Min:  {eval_samples["avg_cpu"].min():.4f}')
print(f'Max:  {eval_samples["avg_cpu"].max():.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(eval_samples['datetime'], eval_samples['avg_mem'],
        color='#9C27B0')
ax.set_xlabel('Time')
ax.set_ylabel('Normalized Memory')
ax.set_title('Memory Utilization During Evaluation')
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig(f'{LOG_DIR}/eval_memory.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Mean: {eval_samples["avg_mem"].mean():.4f}')
print(f'Min:  {eval_samples["avg_mem"].min():.4f}')
print(f'Max:  {eval_samples["avg_mem"].max():.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(eval_samples['datetime'], eval_samples['replicas'],
        color='#3F51B5')
ax.set_xlabel('Time')
ax.set_ylabel('Ready Replicas')
ax.set_title('Replica Count During Evaluation')
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig(f'{LOG_DIR}/eval_replicas.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Mean: {eval_samples["replicas"].mean():.2f}')
print(f'Min:  {eval_samples["replicas"].min()}')
print(f'Max:  {eval_samples["replicas"].max()}')